# ReID Open-Set: Google Colab Runner
Hardened execution runner for training, evaluating, and checkpoint management on Colab T4 GPU.

### Features:
- **Security**: Never puts PAT tokens in clone URLs or .git/config. Colab only pulls.
- **Resilience**: Checkpoints and CSV logs stream directly to `/content/drive/MyDrive/reid_openset/runs/{EXP_NAME}/`. Disconnects lose at most one epoch; re-running auto-resumes.
- **Evaluation Integrity**: Evaluates the final epoch snapshot (`checkpoint_epoch_060.pth`) or latest checkpoint, never a test-set selected "best" model.
- **Modal Execution**: Choose `RUN_MODE` from `("smoke", "train", "eval", "sweep")`.

In [ ]:
# ==============================================================================
# 1. RUNNER PARAMETERS & EXECUTION MODE
# ==============================================================================
# Modes: "smoke" (quick 2-epoch test) | "train" (selected CONFIG) | "eval" (evaluate final snapshot) | "sweep" (all 5 ablations)
RUN_MODE = "smoke"

BRANCH = "main"
# Config choice: configs/baseline.yaml or any configs/exp/e*.yaml
CONFIG = "configs/exp/e0_baseline_val.yaml"
from pathlib import Path
EXP_NAME = Path(CONFIG).stem
REPO_URL = "https://github.com/Shandeepsugumar/reid-openset.git"

# Dataset and Output Directory Paths
DRIVE_MOUNT = "/content/drive"
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/dataset/archive (1).zip"
LOCAL_DATA_DIR = "/content/data/Market-1501-v15.09.15"

# Run outputs stream directly to Drive so disconnects preserve checkpoints and enable seamless auto-resume
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/reid_openset/runs"
EXP_RUN_DIR = f"{DRIVE_OUTPUT_ROOT}/{EXP_NAME}"
LOCAL_REPO_DIR = "/content/reid-openset"

In [ ]:
# ==============================================================================
# 2. GPU VERIFICATION & DRIVE MOUNT
# ==============================================================================
import os
import torch

assert torch.cuda.is_available(), "ERROR: Please enable a GPU runtime (Runtime -> Change runtime type -> T4 GPU)"
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

from google.colab import drive

drive.mount(DRIVE_MOUNT)
os.makedirs(EXP_RUN_DIR, exist_ok=True)
print(f"[Drive] Run directory ready at: {EXP_RUN_DIR}")

In [ ]:
# ==============================================================================
# 3. SECURE GIT CLONE OR PULL (Colab only pulls; never prints or stores tokens)
# ==============================================================================
import base64
import os
import subprocess
from pathlib import Path

# Secure credential retrieval from Colab Secrets (userdata)
# If repo is public, no token is used. If private, token is passed in-memory via extraheader.
pat = None
try:
    from google.colab import userdata
    pat = userdata.get("GITHUB_PAT")
except Exception:
    pass

git_extra_args = []
if pat:
    auth_header = f"AUTHORIZATION: basic {base64.b64encode(f':{pat}'.encode()).decode()}"
    git_extra_args = ["-c", f"http.extraheader={auth_header}"]

if not Path(LOCAL_REPO_DIR).is_dir():
    print(f"[Git] Cloning {REPO_URL} (branch: {BRANCH})...")
    clone_cmd = ["git"] + git_extra_args + ["clone", "-b", BRANCH, REPO_URL, LOCAL_REPO_DIR]
    res = subprocess.run(clone_cmd, capture_output=True, text=True)
    if res.returncode != 0:
        err = res.stderr.replace(pat, "[REDACTED]") if pat else res.stderr
        raise RuntimeError(f"Git clone failed: {err}")
    print("[Git] Cloned successfully.")
else:
    print(f"[Git] Repository exists at {LOCAL_REPO_DIR}. Pulling latest changes from branch '{BRANCH}'...")
    pull_cmd = ["git", "-C", LOCAL_REPO_DIR] + git_extra_args + ["pull", "origin", BRANCH]
    res = subprocess.run(pull_cmd, capture_output=True, text=True)
    if res.returncode != 0:
        err = res.stderr.replace(pat, "[REDACTED]") if pat else res.stderr
        raise RuntimeError(f"Git pull failed: {err}")
    print("[Git] Pull complete.")

%cd {LOCAL_REPO_DIR}
subprocess.run(["git", "log", "-n", "1", "--oneline"], check=True)

In [ ]:
# ==============================================================================
# 4. ENVIRONMENT & PACKAGE SETUP (Editable install without touching torch)
# ==============================================================================
%cd {LOCAL_REPO_DIR}
!pip install --no-deps -e .

In [ ]:
# ==============================================================================
# 5. DATASET EXTRACTION TO /content/data
# ==============================================================================
import os
import zipfile
from pathlib import Path

dataset_root = Path(LOCAL_DATA_DIR)
if not (dataset_root / "bounding_box_train").is_dir():
    assert Path(DRIVE_DATASET_ZIP).is_file(), f"Dataset zip not found at: {DRIVE_DATASET_ZIP}"
    print(f"Extracting {DRIVE_DATASET_ZIP} to /content/data ...")
    os.makedirs("/content/data", exist_ok=True)
    with zipfile.ZipFile(DRIVE_DATASET_ZIP, "r") as z:
        z.extractall("/content/data")
    print(f"Extraction complete: {dataset_root}")
else:
    print(f"Dataset already verified at: {dataset_root}")

In [ ]:
%cd {LOCAL_REPO_DIR}
# ==============================================================================
# 6. [MODE: smoke] FAST SMOKE TEST (2 Epochs, 50 Identities)
# ==============================================================================
if RUN_MODE == "smoke":
    print(f"--- RUNNING SMOKE TEST (RUN_MODE='{RUN_MODE}') ---")
    !python scripts/train.py --config {CONFIG} --smoke-test --data-root {LOCAL_DATA_DIR} --output-root {DRIVE_OUTPUT_ROOT}
else:
    print(f"Skipping smoke test (current RUN_MODE is '{RUN_MODE}')")

In [ ]:
%cd {LOCAL_REPO_DIR}
# ==============================================================================
# 7. [MODE: train] FULL BASELINE TRAINING (60 Epochs with Auto-Resume)
# ==============================================================================
# Checkpoints and metrics stream directly to Drive under:
# /content/drive/MyDrive/reid_openset/runs/{EXP_NAME}/
# If Colab disconnects, re-running this cell automatically resumes from latest_checkpoint.pth.
if RUN_MODE == "train":
    print(f"--- STARTING FULL TRAINING (RUN_MODE='{RUN_MODE}') ---")
    !python scripts/train.py --config {CONFIG} --data-root {LOCAL_DATA_DIR} --output-root {DRIVE_OUTPUT_ROOT}
else:
    print(f"Skipping training (current RUN_MODE is '{RUN_MODE}')")

In [ ]:
%cd {LOCAL_REPO_DIR}
# ==============================================================================
# 8. [MODE: eval] EVALUATION ON FINAL EPOCH SNAPSHOT
# ==============================================================================
# The baseline is evaluated from the FINAL epoch snapshot (checkpoint_epoch_060.pth)
# or latest checkpoint in Drive, NEVER from a test-set selected "best" checkpoint.
if RUN_MODE in ("train", "eval"):
    print(f"--- RUNNING EVALUATION (RUN_MODE='{RUN_MODE}') ---")
    final_snapshot = f"{EXP_RUN_DIR}/checkpoints/checkpoint_epoch_060.pth"
    latest_ckpt = f"{EXP_RUN_DIR}/checkpoints/latest_checkpoint.pth"
    eval_ckpt = final_snapshot if os.path.isfile(final_snapshot) else latest_ckpt
    
    !python scripts/evaluate.py --config {CONFIG} --checkpoint "{eval_ckpt}" --data-root {LOCAL_DATA_DIR} --output-root {DRIVE_OUTPUT_ROOT}
else:
    print(f"Skipping evaluation (current RUN_MODE is '{RUN_MODE}')")

In [ ]:
# ==============================================================================
# 9. [MODE: sweep] ABLATION SWEEP OVER EXPERIMENTS
# ==============================================================================
# Iterates over e1, e2, e3 in order: trains each (60 epochs), evaluates on test set,
# and aggregates results into Drive ablations summary table.
import os
from pathlib import Path

# Ensure working directory is repo root
%cd /content/reid-openset

if RUN_MODE == "sweep":
    print(f"--- STARTING ABLATION SWEEP (RUN_MODE='{RUN_MODE}') ---")
    sweep_configs = [
        "configs/exp/e1_cosface.yaml",
        "configs/exp/e2_cosface_hardid.yaml",
        "configs/exp/e3_cosface_hardid_absneg.yaml",
        # Uncomment below if e0 or e1b also need to be run in this sweep:
        # "configs/exp/e0_baseline_val.yaml",
        # "configs/exp/e1b_arcface.yaml",
    ]
    for cfg in sweep_configs:
        exp_id = Path(cfg).stem
        print(f"\n{'='*60}\n>>> SWEEP: Training {exp_id} ({cfg})\n{'='*60}")
        !cd /content/reid-openset && python scripts/train.py --config {cfg} --data-root {LOCAL_DATA_DIR} --output-root {DRIVE_OUTPUT_ROOT}
        
        # Test evaluation strictly on the final epoch snapshot
        final_ckpt = f"{DRIVE_OUTPUT_ROOT}/{exp_id}/checkpoints/checkpoint_epoch_060.pth"
        latest_ckpt = f"{DRIVE_OUTPUT_ROOT}/{exp_id}/checkpoints/latest_checkpoint.pth"
        eval_ckpt = final_ckpt if os.path.isfile(final_ckpt) else latest_ckpt
        print(f"\n>>> SWEEP: Evaluating {exp_id} with checkpoint {eval_ckpt}")
        !cd /content/reid-openset && python scripts/evaluate.py --config {cfg} --checkpoint "{eval_ckpt}" --data-root {LOCAL_DATA_DIR} --output-root {DRIVE_OUTPUT_ROOT}
    
    # Run aggregation across all completed runs
    print("\n>>> SWEEP: Aggregating ablation results...")
    !cd /content/reid-openset && python scripts/aggregate_results.py --output-root {DRIVE_OUTPUT_ROOT} --doc-path {DRIVE_OUTPUT_ROOT}/ablations.md --csv-path {DRIVE_OUTPUT_ROOT}/ablations.csv
    print(f"\n[Sweep Complete] Ablation summary written to {DRIVE_OUTPUT_ROOT}/ablations.md and ablations.csv")
else:
    print(f"Skipping ablation sweep (current RUN_MODE is '{RUN_MODE}')")


In [ ]:
# ==============================================================================
# 10. VERIFY RUN ARTIFACTS ON GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

run_path = Path(EXP_RUN_DIR)
if run_path.is_dir():
    print(f"Artifacts preserved on Google Drive at: {run_path}")
    for item in sorted(run_path.rglob("*")):
        if item.is_file():
            size_mb = item.stat().st_size / (1024 * 1024)
            print(f"  - {item.relative_to(run_path)} ({size_mb:.2f} MB)")
else:
    print(f"No run outputs found yet at: {run_path}")